02 - H2: Transfer statt Aufbau - Hauptgruppen-Drilldown.

Hauptbefund: Eigene Bundes-Investitionen (UG 81/82) stagnieren. Investitions-
zuschuesse an Dritte (UG 83/86/87/88/89) verdoppeln ihren Anteil.

Drilldowns:
  1. Konsumtiv vs. Investiv ueber Jahre
  2. Investiv-Detail: eigene vs. Transfer
  3. Treiber-Identifikation (welche Einzelplaene, welche Titel)
  4. Robustheit weite Abgrenzung
  5. HG-6-Spin-off: Mikroelektronik-Peak 2023

Outputs: figures/h2_schere_eigene_vs_transfer.png

In [1]:
import sys
from pathlib import Path
_root = (
    Path(__file__).parent.parent if '__file__' in globals() else
    next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'src').exists())
)
sys.path.insert(0, str(_root))
__import__('os').chdir(_root)

import pandas as pd
from src.load import load, KATS

df = load('data/raw/Digitalhaushalt_Open_Data.xlsx')

# Nur digitale Titel mit eng>0
d = df[(df['any_tag'] == 1) & (df['digi_soll_eng'] > 0)].copy()

In [2]:
print("=== Konsumtiv (HG 4-6) vs. Investiv (HG 7-8) - digi_soll_eng (Mrd) ===")
d['typ_kons_inv'] = d['hg'].map(
    lambda x: 'konsumtiv' if x in ['4', '5', '6']
    else ('investiv' if x in ['7', '8'] else 'sonstiges')
)
agg = d.groupby(['jahr', 'typ_kons_inv'])['digi_soll_eng'].sum().div(1e6).unstack()
agg['Sigma'] = agg.sum(axis=1)
for c in ['konsumtiv', 'investiv', 'sonstiges']:
    if c in agg.columns:
        agg[f'{c}_%'] = (agg[c] / agg['Sigma'] * 100).round(1)
print(agg.round(2))

=== Konsumtiv (HG 4-6) vs. Investiv (HG 7-8) - digi_soll_eng (Mrd) ===
typ_kons_inv  investiv  konsumtiv  Sigma  konsumtiv_%  investiv_%
jahr                                                             
2019              2.32       6.18   8.51         72.7        27.3
2021              4.34      12.21  16.55         73.8        26.2
2023              5.03      14.16  19.19         73.8        26.2
2024              6.85      11.09  17.94         61.8        38.2


In [3]:
print("\n=== HG 8 Detail: eigene vs. Transfer-Investitionen (Mrd) ===")
hg8 = d[d['hg'] == '8']
piv = hg8.groupby(['ug', 'jahr'])['digi_soll_eng'].sum().div(1e6).unstack('jahr').fillna(0)
piv['Delta_19_24'] = piv[2024] - piv[2019]
piv = piv.sort_values(2024, ascending=False)
print(piv.round(3).to_string())

# Aggregiert nach Invest-Typ
print("\n=== Aggregiert: eigene Invest (UG 81/82) vs. Transfer-Invest (UG 83/86/87/88/89) ===")
inv_agg = (
    d[d['typ_invest'].notna()]
    .groupby(['jahr', 'typ_invest'])['digi_soll_eng']
    .sum().div(1e6).unstack().fillna(0)
)
inv_agg['Sigma_Digital'] = d.groupby('jahr')['digi_soll_eng'].sum().div(1e6)
inv_agg['eigene_%'] = (inv_agg['A_eigene_Invest'] / inv_agg['Sigma_Digital'] * 100).round(1)
inv_agg['transfer_%'] = (inv_agg['B_Transfer_Invest'] / inv_agg['Sigma_Digital'] * 100).round(1)
print(inv_agg.round(2).to_string())


=== HG 8 Detail: eigene vs. Transfer-Investitionen (Mrd) ===
jahr   2019   2021   2023   2024  Delta_19_24
ug                                           
89    0.861  2.443  3.425  4.164        3.303
88    0.302  0.194  0.287  1.528        1.226
81    0.927  1.419  1.239  1.084        0.157
83    0.191  0.251  0.048  0.020       -0.171

=== Aggregiert: eigene Invest (UG 81/82) vs. Transfer-Invest (UG 83/86/87/88/89) ===
typ_invest  A_eigene_Invest  B_Transfer_Invest  Sigma_Digital  eigene_%  transfer_%
jahr                                                                               
2019                   0.93               1.35           8.51      10.9        15.9
2021                   1.42               2.89          16.55       8.6        17.5
2023                   1.24               3.76          19.19       6.5        19.6
2024                   1.08               5.71          17.94       6.0        31.8


In [4]:
print("\n=== Top-Treiber des Transfer-Booms (UG 88+89, Delta 2019->2024 Mrd) ===")
tr = d[d['ug'].isin(['88', '89'])]
piv2 = (
    tr.groupby(['einzelplan', 'einzelplantext', 'jahr'])['digi_soll_eng']
    .sum().div(1e6).unstack('jahr').fillna(0)
)
piv2['Delta_19_24'] = piv2[2024] - piv2[2019]
print(piv2.sort_values('Delta_19_24', ascending=False).head(6).round(2).to_string())


=== Top-Treiber des Transfer-Booms (UG 88+89, Delta 2019->2024 Mrd) ===
jahr                                                                    2019  2021  2023  2024  Delta_19_24
einzelplan einzelplantext                                                                                  
12         Bundesministerium für Digitales und Verkehr                  0.00  0.00  1.68  3.31         3.31
30         Bundesministerium für Bildung und Forschung                  0.37  0.26  0.39  1.72         1.35
6          Bundesministerium des Innern und für Heimat                  0.00  0.00  0.29  0.25         0.25
9          Bundesministerium für Wirtschaft und Klimaschutz             0.00  0.00  1.18  0.25         0.25
25         Bundesministerium für Wohnen, Stadtentwicklung und Bauwesen  0.00  0.00  0.13  0.13         0.13
8          Bundesministerium der Finanzen                               0.00  0.00  0.01  0.01         0.01


In [5]:
print("\n=== Robustheit: gleiche Struktur unter digi_soll_weit ===")
dw = df[(df['any_tag'] == 1) & (df['digi_soll_weit'] > 0)].copy()
dw['typ_fein'] = dw['ug'].map(
    lambda x: 'A_eigene' if x in ['81', '82']
    else ('B_transfer' if x in ['83', '86', '87', '88', '89']
          else ('konsumtiv' if x[:1] in ['4', '5', '6'] else 'sonstiges'))
)
agg_w = dw.groupby(['jahr', 'typ_fein'])['digi_soll_weit'].sum().div(1e6).unstack().round(2)
print(agg_w.to_string())


=== Robustheit: gleiche Struktur unter digi_soll_weit ===
typ_fein  A_eigene  B_transfer  konsumtiv  sonstiges
jahr                                                
2019          0.93        1.45       7.21       0.04
2021          1.43        3.00      13.33       0.03
2023          1.24        3.89      15.38       0.04
2024          1.09        5.86      12.09       0.05


In [6]:
print("\n=== HG-6-Peak 2023: Was steckt dahinter? ===")
hg6 = d[d['hg'] == '6'].copy()
# Mikroelektronik-Titel isolieren
is_mikro = (
    (hg6['einzelplan'] == 60)
    & (hg6['gruppe'] == 686)
    & (hg6['titel_text'].fillna('').str.contains('Mikroelektronik', case=False))
)
print(f"Mikroelektronik-Titel (EP 60, Gruppe 686): {is_mikro.sum()} Zeilen")
print("Volumen je Jahr (Mrd):")
print(hg6[is_mikro].groupby('jahr')['digi_soll_eng'].sum().div(1e6).round(3).to_string())

print("\nHG 6 mit/ohne Mikroelektronik-Titel (Mrd):")
total_hg6 = hg6.groupby('jahr')['digi_soll_eng'].sum().div(1e6)
ohne = hg6[~is_mikro].groupby('jahr')['digi_soll_eng'].sum().div(1e6)
out = pd.DataFrame({'HG6_total': total_hg6, 'HG6_ohne_Mikro': ohne}).round(2)
print(out.to_string())


=== HG-6-Peak 2023: Was steckt dahinter? ===
Mikroelektronik-Titel (EP 60, Gruppe 686): 1 Zeilen
Volumen je Jahr (Mrd):
jahr
2023    2.74

HG 6 mit/ohne Mikroelektronik-Titel (Mrd):
      HG6_total  HG6_ohne_Mikro
jahr                           
2019       2.61            2.61
2021       5.99            5.99
2023       7.97            5.23
2024       4.60            4.60
